In [4]:
import os
import sys
import time
import yaml
import pandas as pd
import json
from string import Template

with open('../../config.local.yaml', 'r') as f:
    local_config = yaml.safe_load(f)

LOCAL_PATH = local_config['LOCAL_PATH']
DATA_PATH = local_config['DATA_PATH']

sys.path.append(os.path.join(LOCAL_PATH, "src/python"))

from utils import is_casenum, extract_json

In [5]:
meetings_df = pd.read_csv(os.path.join(DATA_PATH, "intermediate_data/cpc/meetings-manifest.csv"))
DATES = sorted(list(meetings_df['date']))

In [7]:
reasons_df = {}

for idx, row in meetings_df.iterrows():
    date = row['date']
    year = date[0:4]
    input_filepath = os.path.join(DATA_PATH, "intermediate_data/cpc", year, date, "supplemental-docs-summarized-2.json")
    if not os.path.exists(input_filepath):
        continue
    with open(input_filepath, 'r', encoding='utf-8') as f:
        j = json.load(f)

    for doc in j:
        for item_no, reasons in doc["support_or_oppose_reasons"].items():
            for reason in reasons:
                reasons_df[reason] = reasons_df.get(reason, 0) + 1

reasons_df = pd.DataFrame(list(reasons_df.items()), columns=["reason", "count"])
reasons_df = reasons_df.sort_values(by="count", ascending=False)

reasons_df.to_csv(os.path.join(DATA_PATH, "intermediate_data/cpc/reasons.csv"), index=False)